# Week 2: Baseline Models & CV Setup

This notebook covers the second week of the PM2.5 forecasting project, focusing on establishing a robust Time-Series Cross-Validation strategy and training baseline models (SARIMA and Prophet).

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Time-Series Cross Validation (Expanding Window)
# Ensuring no future data leakage during baseline training
tscv = TimeSeriesSplit(n_splits=3)
# Usage during training: for train_index, test_index in tscv.split(df): ...

In [ ]:
# Baseline 1: SARIMA
def run_sarima_baseline(train_series, test_series):
    print('Training SARIMA Baseline...')
    model = SARIMAX(train_series, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0))
    results = model.fit(disp=False)

    preds = results.forecast(steps=len(test_series))
    rmse = np.sqrt(mean_squared_error(test_series, preds))
    mae = mean_absolute_error(test_series, preds)

    print(f'SARIMA Baseline -> RMSE: {rmse:.2f}, MAE: {mae:.2f}')
    return preds

In [ ]:
# Baseline 2: Facebook Prophet
def run_prophet_baseline(train_df, periods):
    # Prophet expects columns strictly named 'ds' (datetime) and 'y' (target)
    print('Training Prophet Baseline...')
    m = Prophet(daily_seasonality=False, yearly_seasonality=True)
    m.fit(train_df)

    future = m.make_future_dataframe(periods=periods)
    forecast = m.predict(future)

    return forecast[['ds', 'yhat']].tail(periods)